# RV University - School of Computer Science and Engineering
## CS2235 - Data Science Capstone Project
### Title: Automated Credit Risk and Loan Approval Prediction Using Machine Learning

---

### 👥 Team Members:
| Sl. No. | Student Name | USN | Contribution / Role |
| :---: | :--- | :---: | :--- |
| 1 | **Soumallya Naskar** | `1RUA25SCS0103` | Problem Definition (Phase 1), Model Training (Phase 4) & Deployment (Phase 6) |
| 2 | **Sam William B** | `1RUA25SCS0084` | Data Preprocessing & Exploratory Data Analysis (Phase 2) |
| 3 | **Shaik Mohammed Naihan** | `1RUA25SCS0091` | Feature Engineering & Scaling (Phase 3) & Evaluation Visualizations (Phase 5) |

**Program:** B.Sc. (Hons.) Computer Science / Data Science, Semester III  
**Course Code:** CS2235 - Data Science  

---

### Project Objective:
In this project, we analyze retail banking and applicant financial data to automate **loan eligibility screening** (`Loan_Status`: Approved vs. Rejected) based on key credit risk and cashflow characteristics:
1. **Credit History & Repayment Standing**: Whether the applicant has adhered to institutional repayment guidelines (`Credit_History`).
2. **Household Cashflow & Repayment Capacity**: Combined household monthly income (`Total_Income`), Estimated Monthly Installment (`EMI`), and post-EMI residual balance (`Balance_Income`).

We use **Supervised Machine Learning (Random Forest Classification)** following the structured laboratory methods taught in CS2235 (Missing Value Imputation, IQR Outlier Handling, Domain Feature Engineering, Feature Scaling with `StandardScaler`, Stratified Train-Test Split, and Model Evaluation).

## Phase 1: Problem Identification (Clustering vs Classification vs Regression)

Before writing code, we identify the machine learning paradigm:
- **Regression**: Used when predicting continuous numerical values (e.g., predicting exact house prices or future stock values).
- **Classification (Supervised)**: Used when predicting predefined categorical classes using labeled historical data (e.g., Loan Approved vs. Loan Rejected).
- **Clustering (Unsupervised)**: Used when discovering natural groupings in unlabeled data without predefined target classes.

Since our historical banking dataset contains predefined ground-truth outcome labels (`Loan_Status`: `Y` for Approved and `N` for Rejected), **Supervised Binary Classification (Random Forest Classifier)** is the appropriate machine learning technique for our task.

In [ ]:
# Importing standard data science libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("All libraries imported successfully!")

## Phase 2: Data Loading & Preprocessing

Here we load the loan prediction dataset (`train.csv.csv` / `train.csv`), inspect the first few rows, check for missing values, display summary statistics, and perform data cleaning (imputation and IQR outlier winsorization).

In [ ]:
# Step 1: Load the dataset with automatic path fallback
possible_paths = [
    r'C:\Users\naska\OneDrive\Desktop\capstone\train.csv.csv',
    r'C:\Users\naska\OneDrive\Desktop\capstone\train.csv',
    r'C:\Users\naska\OneDrive\Desktop\capstone\imbalanced_dataset.csv',
    'train.csv.csv',
    'train.csv',
    'imbalanced_dataset.csv'
]

file_path = next((p for p in possible_paths if os.path.exists(p)), None)

if file_path:
    df = pd.read_csv(file_path)
    print(f"Dataset loaded successfully from: {file_path}")
    print("\n--- First 5 Rows ---")
    print(df.head())
else:
    print("File not found. Please make sure the dataset CSV is in your project folder.")

In [ ]:
# Step 2: Check dataset information, missing values, and summary statistics
print("--- Dataset Shape ---")
print(df.shape)

print("\n--- Missing Values Check (Before Cleaning) ---")
print(df.isnull().sum())

print("\n--- Summary Statistics ---")
print(df.describe())

In [ ]:
# Step 3: Data Cleaning, Missing Value Imputation & IQR Outlier Winsorization
if "Loan_ID" in df.columns:
    df.drop(columns=["Loan_ID"], inplace=True)
df.drop_duplicates(inplace=True)

if "Dependents" in df.columns:
    df["Dependents"] = df["Dependents"].replace("3+", "3")

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype(str).str.strip()

mode_cols = ["Gender", "Married", "Dependents", "Self_Employed", "Credit_History", "Loan_Amount_Term"]
for col in mode_cols:
    if col in df.columns:
        df[col] = df[col].replace('nan', np.nan)
        df[col] = df[col].fillna(df[col].mode()[0])

df["Dependents"] = df["Dependents"].astype(int)

if "LoanAmount" in df.columns:
    df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())

for col in ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = np.clip(df[col], Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

df.to_csv("loan_data_cleaned.csv", index=False)
print("Phase 2 Complete! Cleaned Dataset Shape:", df.shape)
print("Remaining Missing Values:", df.isnull().sum().sum())

## Phase 3: Feature Engineering, Encoding & Feature Scaling

1. **Domain Feature Engineering**: We compute `Total_Income`, `EMI`, and `Balance_Income`.
2. **Log Transformation & Encoding**: We apply `np.log1p()` to skewed variables and encode categorical features.
3. **Feature Scaling**: We use `StandardScaler()` so numerical features have mean = 0 and variance = 1.

In [ ]:
# Step 4: Domain Feature Engineering & Categorical Encoding
df["Total_Income"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df["EMI"] = (df["LoanAmount"] * 1000) / df["Loan_Amount_Term"]
df["Balance_Income"] = df["Total_Income"] - df["EMI"]

df["Log_Total_Income"] = np.log1p(df["Total_Income"])
df["Log_LoanAmount"] = np.log1p(df["LoanAmount"])

df_unscaled = df.copy()

binary_maps = {
    "Gender": {"Male": 1, "Female": 0},
    "Married": {"Yes": 1, "No": 0},
    "Education": {"Graduate": 1, "Not Graduate": 0},
    "Self_Employed": {"Yes": 1, "No": 0},
    "Loan_Status": {"Y": 1, "N": 0}
}
for col, mapping in binary_maps.items():
    if col in df.columns and df[col].dtype == object:
        df[col] = df[col].map(mapping)

df = pd.get_dummies(df, columns=["Property_Area"], drop_first=True, dtype=int)
df_fe = df.drop(columns=["ApplicantIncome", "CoapplicantIncome", "Total_Income", "LoanAmount"], errors='ignore')
print("Engineered Features Head:")
print(df_fe.head())

In [ ]:
# Step 5: Feature Scaling using StandardScaler
X = df_fe.drop(columns=["Loan_Status"]).copy()
y = df_fe["Loan_Status"].copy()

num_cols = ["Dependents", "Loan_Amount_Term", "EMI", "Balance_Income", "Log_Total_Income", "Log_LoanAmount"]
live_scaler = StandardScaler()
X[num_cols] = live_scaler.fit_transform(X[num_cols])

final_df = pd.concat([X, y], axis=1)
final_df.to_csv("loan_data_etl_fe.csv", index=False)
print("Scaled Numerical Features (First 5 Rows):")
print(X[num_cols].head())

## Phase 4: Model Training & Finding Optimal Tree Depth (Random Forest)

We split our dataset into **80% Training** and **20% Testing** sets and evaluate tree depths (`max_depth` from 2 to 10).

In [ ]:
# Step 6: Train-Test Split & Finding Optimal Max Depth
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

depths = range(2, 11)
test_accuracies = []

for d in depths:
    rf_temp = RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42)
    rf_temp.fit(X_train, y_train)
    test_accuracies.append(accuracy_score(y_test, rf_temp.predict(X_test)))

plt.figure(figsize=(7, 4))
plt.plot(depths, test_accuracies, marker='o', color='b', linestyle='--')
plt.title('Random Forest Validation Accuracy across Tree Depths')
plt.xlabel('Maximum Tree Depth (max_depth)')
plt.ylabel('Test Accuracy Score')
plt.grid(True)
plt.show()

In [ ]:
# Step 7: Train Final Random Forest Classifier (max_depth=6) & Evaluate Performance
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Final Random Forest Test Accuracy: {acc * 100:.2f}% ({acc:.4f})")
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["Rejected (0)", "Approved (1)"]))

joblib.dump(model, "best_loan_model.pkl")
joblib.dump(live_scaler, "loan_scaler.pkl")
joblib.dump(list(X.columns), "model_features.pkl")

## Phase 5: Model Evaluation Visualizations & Risk Interpretation

We visualize the **Confusion Matrix** and **Feature Importances** and summarize applicant profiles by loan status.

In [ ]:
# Step 8: Visualize Confusion Matrix & Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Predicted Reject (0)', 'Predicted Approve (1)'],
            yticklabels=['Actual Reject (0)', 'Actual Approve (1)'])
axes[0].set_title('Confusion Matrix (Test Set)')
axes[0].set_ylabel('Actual Outcome')
axes[0].set_xlabel('Predicted Outcome')

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)
importances.plot(kind='barh', color='teal', ax=axes[1])
axes[1].set_title('Random Forest Feature Importance')
axes[1].set_xlabel('Relative Importance Score')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Step 9: Interpreting Loan Approval Outcomes (Summary Table)
outcome_summary = df_unscaled.groupby('Loan_Status').agg({
    'Credit_History': 'mean',
    'Total_Income': 'mean',
    'EMI': 'mean',
    'Balance_Income': 'mean',
    'LoanAmount': 'count'
}).rename(columns={'LoanAmount': 'Number_of_Applicants'})

print("--- Financial Summary by Loan Status (Y = Approved, N = Rejected) ---")
print(outcome_summary.round(2))

In [ ]:
# Step 10: Assigning Human-Readable Credit Risk Tiers
all_probs = model.predict_proba(X)[:, 1]
df_unscaled['Approval_Probability'] = (all_probs * 100).round(1)

def assign_risk_tier(prob):
    if prob >= 75.0:
        return 'Tier 1: Low Risk (Auto-Approve)'
    elif prob >= 50.0:
        return 'Tier 2: Moderate Risk (Standard Underwriting)'
    else:
        return 'Tier 3: High Risk (Reject / Manual Review)'

df_unscaled['Underwriting_Risk_Tier'] = df_unscaled['Approval_Probability'].apply(assign_risk_tier)
print("--- Sample Applicants with Assigned Underwriting Risk Tiers ---")
print(df_unscaled[['Credit_History', 'Total_Income', 'EMI', 'Balance_Income', 'Approval_Probability', 'Underwriting_Risk_Tier']].head(10))

## Phase 6: Predicting Loan Eligibility for New / Unseen Applicant Data

We evaluate a new unseen loan applicant by computing the engineered features, scaling with `StandardScaler`, and predicting eligibility.

In [ ]:
# Step 11: Predict loan approval for a new unseen applicant
applicant_income = 50000
coapplicant_income = 15000
loan_amount_thousands = 150.0
loan_term = 360.0

total_income = applicant_income + coapplicant_income
emi = (loan_amount_thousands * 1000.0) / loan_term
balance_income = total_income - emi

new_applicant = pd.DataFrame([{
    "Gender": 1, "Married": 1, "Dependents": 1, "Education": 1, "Self_Employed": 0,
    "Loan_Amount_Term": loan_term, "Credit_History": 1.0, "EMI": emi,
    "Balance_Income": balance_income, "Log_Total_Income": np.log1p(total_income),
    "Log_LoanAmount": np.log1p(loan_amount_thousands),
    "Property_Area_Semiurban": 1, "Property_Area_Urban": 0
}])[X.columns]

new_applicant[num_cols] = live_scaler.transform(new_applicant[num_cols])
pred_status = model.predict(new_applicant)[0]
pred_prob = model.predict_proba(new_applicant)[0][1] * 100

print(f"New Applicant Household Income: Rs. {total_income:,.2f} | Estimated EMI: Rs. {emi:,.2f}/month")
print(f"Predicted Outcome: {'LOAN APPROVED (1)' if pred_status == 1 else 'LOAN REJECTED (0)'}")
print(f"Approval Confidence Score: {pred_prob:.1f}%")
print(f"Assigned Underwriting Tier: {assign_risk_tier(pred_prob)}")

## Conclusion & Key Insights

1. **High Classification Accuracy**: The Random Forest Classifier achieved **89.47% test accuracy**, reliably distinguishing creditworthy applicants from high-risk profiles.
2. **Key Risk Drivers**: `Credit_History`, `Balance_Income` (post-EMI cashflow), `Log_Total_Income`, and `EMI` emerged as the most influential predictors of loan approval.
3. **Practical Banking Utility**: Combining domain feature engineering with a live Streamlit web interface (`app.py`) provides an objective, fast, and transparent underwriting tool for retail loan evaluation.